# Anima LoRA 学習ノートブック (完全自動・簡単テンプレート・sd-scripts版) v1.0

**Anima (CircleStone Labs 2B DiT)** の LoRA を Google Colab 上で完全自動学習するノートブックです。
`sd-scripts` (`kohya-ss`) の公式 Anima 学習エンジンを採用しており、学習設定は制作者推奨の安定設定そのままです。
LanPaint や ComfyUI での生成・インペイントに完全対応した LoRA (`net.` プレフィックス付き) が出力されます。

---

### 🌟 簡単テンプレート (設定ファイル不要・フォルダに入れるだけ)
あなたが操作するのは **最初の3セルだけ** (トークン入力・Drive許可・キュー確認) です。
`jobs.json` などの設定ファイルは**一切不要**です。

```
Google Drive フォルダ構成:
 📂 MyDrive/anima/queue/    <-- 学習させたいZIPをここに入れるだけ (自動作成)
 📦 MyDrive/anima/done/     <-- 正常終了したZIPが自動的にここに退避
 💾 MyDrive/anima/outputs/  <-- 完成したLoRAがここに自動納品
```

- トリガーワードは ZIP 内のキャプション先頭タグ (またはファイル名) から自動検出
- ステップ数は画像枚数から自動決定 (枚数に応じた最適値: 30枚なら1000steps)
- 複数ZIPを入れておけば順番に連続学習します

**学習パラメータ** (Anima 推奨値そのまま):
- 基本アーキテクチャ: MiniTrainDIT (2B DiT) + Qwen3-0.6B + WanVAE (16ch)
- ネットワーク: `networks.lora_anima` (rank=16, alpha=16, DiT学習)
- 学習率 (lr): `1e-4` (AdamW8bit / constant)
- フローシフト: `discrete_flow_shift = 3.0` / `timestep_sample_method = logit_normal`
- 最適化: `mixed_precision = bf16`, latents & text_encoder事前キャッシュ, 勾配チェックポイント
- 必要VRAM: 約8〜10GB (T4 / L4 / A100 すべてで快適に動作します)


## 1. あなたの操作 (ここだけ・最初に全部済ませる)

以下の3つのセルを上から順に実行してください。

### 1-1. HuggingFaceトークン
Colab 左側の 🔑 (シークレット) に `HF_TOKEN` が登録されていれば自動読み込みします。未登録の場合は入力欄が出ます。

In [ ]:
import getpass, os
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    print("Colab SecretからHF_TOKENを読み込みました")
except Exception:
    token = os.environ.get('HF_TOKEN') or getpass.getpass('HuggingFaceトークンを入力 (hf_...): ')
os.environ['HF_TOKEN'] = token
assert token.startswith('hf_'), 'トークンの形式が不正です (hf_で始まる必要があります)'
print('HuggingFaceトークン設定OK')


### 1-2. Google Driveマウント
データセット読み込みとLoRA保存のために Google Drive を接続します。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google DriveマウントOK')


### 1-3. キュー確認 (設定ファイル不要・queueに入れるだけ)
`MyDrive/anima/queue/` に置かれたZIPファイルを順番に学習します。
正常終了したZIPは自動的に `MyDrive/anima/done/` に移動します。

In [ ]:
# ============================================================
# 1-3. キュー確認 (設定ファイル不要・queueに入れるだけ)
# フォルダ構成:
#  - 学習待ち: MyDrive/anima/queue/
#  - 学習完了: MyDrive/anima/done/
#  - LoRA出力: MyDrive/anima/outputs/
# ============================================================
import glob, os, re, shutil, zipfile

BASE_DIR   = "/content/drive/MyDrive/anima"
QUEUE_DIR  = os.path.join(BASE_DIR, "queue")
DONE_DIR   = os.path.join(BASE_DIR, "done")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")

os.makedirs(QUEUE_DIR, exist_ok=True)
os.makedirs(DONE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# もし BASE_DIR 直下に ZIP が置かれていれば queue に自動移動
for z in glob.glob(os.path.join(BASE_DIR, "*.zip")):
    shutil.move(z, os.path.join(QUEUE_DIR, os.path.basename(z)))

# デフォルト学習パラメータ (Anima 推奨値そのまま)
RESOLUTION = 1024
LR         = 1e-4
RANK       = 16
STEPS      = None  # 未指定なら calc_steps() が画像枚数から自動決定

def extract_trigger_from_zip(zip_path):
    try:
        with zipfile.ZipFile(zip_path, "r") as z:
            txt_files = sorted([f for f in z.namelist() if f.endswith(".txt") and not f.startswith("__MACOSX")])
            if txt_files:
                sample_txt = z.read(txt_files[0]).decode("utf-8", errors="ignore").strip()
                first_tag = sample_txt.split(",")[0].strip()
                if first_tag and len(first_tag.split()) <= 4 and "\n" not in first_tag and "\r" not in first_tag:
                    return first_tag
    except Exception:
        pass
    base = os.path.basename(zip_path).replace(".zip", "")
    clean = re.sub(r"_\d{8}(?:_\d+)?$", "", base)
    clean = re.sub(r"_\d+$", "", clean)
    return clean if clean.endswith("_style") else f"{clean}_style"

zip_files = sorted(glob.glob(os.path.join(QUEUE_DIR, "*.zip")))
assert zip_files, f"学習待ちのZIPが見つかりません。\nGoogle Drive の \"{QUEUE_DIR}\" に学習したいZIPファイルを配置してください。"

JOBS = []
for z in zip_files:
    trig = extract_trigger_from_zip(z)
    JOBS.append({"trigger": trig, "zip": z, "steps": STEPS, "lr": LR, "rank": RANK})

steps_disp = STEPS if STEPS else "AUTO(画像枚数ベース: calc_steps)"
print("="*60)
print(f"🚀 キュー内の学習対象: 計 {len(JOBS)} 件")
print(f"📂 キューフォルダ: {QUEUE_DIR}")
print(f"📦 完了移動先: {DONE_DIR}")
print(f"💾 出力先: {OUTPUT_DIR}")
print(f"⚙️ 設定: 解像度 {RESOLUTION}px / ステップ: {steps_disp} / lr: {LR} / rank: {RANK}")
print("-" * 60)
for i, j in enumerate(JOBS, 1):
    print(f" [{i}] trigger: '{j['trigger']}' | zip: '{os.path.basename(j['zip'])}'")
print("="*60)
print("問題なければメニュー「ランタイム」→「以降のセルを実行」を押してください。\n")


## 2. 環境構築 (自動・一度実行すればスキップ)

`kohya-ss/sd-scripts` リポジトリをクローンし、必要なライブラリをセットアップします。

In [ ]:
import os, sys

print("Python:", sys.version.split()[0])
if not os.path.exists("/content/.anima_env_done"):
    if not os.path.exists("/content/sd-scripts"):
        os.system("git clone https://github.com/kohya-ss/sd-scripts.git /content/sd-scripts")
        print("sd-scripts クローン完了")
    else:
        os.system("cd /content/sd-scripts && git pull")
        print("sd-scripts 最新版取得完了")

    # 依存ライブラリインストール
    os.system("pip install -q accelerate transformers diffusers sentencepiece toml safetensors torchvision ftfy bitsandbytes rich voluptuous imagesize")
    os.system("pip install -q -e /content/sd-scripts")

    open("/content/.anima_env_done", "w").write("done")
    print("環境構築完了 (.anima_env_done 作成)")
else:
    print("環境構築スキップ (既に完了しています)")


## 3. モデルダウンロード (自動・DL済みならスキップ)

Animaの3大必須コンポーネント (DiT, WanVAE, Qwen3-0.6B) をダウンロードします。
すべて公開リポジトリから取得するため、承認待ちなく高速に取得できます (計約6GB / 約1分)。

In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download, snapshot_download

os.makedirs("/content/models/anima", exist_ok=True)
os.makedirs("/content/models/qwen3_06b", exist_ok=True)

# 1. Anima DiT ベースモデル (anima-preview.safetensors)
DIT_PATH = "/content/models/anima/anima-preview.safetensors"
if not os.path.exists(DIT_PATH):
    print("Anima DiT ダウンロード中...")
    p = hf_hub_download("circlestone-labs/Anima", "split_files/diffusion_models/anima-preview.safetensors")
    shutil.copy(p, DIT_PATH)
    print(f"DiT DL完了: {DIT_PATH} ({os.path.getsize(DIT_PATH) / 1e9:.2f} GB)")
else:
    print(f"DiT DL済み: {DIT_PATH}")

# 2. WanVAE (qwen_image_vae.safetensors)
VAE_PATH = "/content/models/anima/qwen_image_vae.safetensors"
if not os.path.exists(VAE_PATH):
    print("Anima VAE (WanVAE) ダウンロード中...")
    p = hf_hub_download("circlestone-labs/Anima", "split_files/vae/qwen_image_vae.safetensors")
    shutil.copy(p, VAE_PATH)
    print(f"VAE DL完了: {VAE_PATH} ({os.path.getsize(VAE_PATH) / 1e6:.1f} MB)")
else:
    print(f"VAE DL済み: {VAE_PATH}")

# 3. Qwen3-0.6B テキストエンコーダー
QWEN3_PATH = "/content/models/qwen3_06b"
if not os.path.exists(os.path.join(QWEN3_PATH, "model.safetensors")):
    print("Qwen3-0.6B テキストエンコーダー ダウンロード中...")
    snapshot_download("Qwen/Qwen3-0.6B", local_dir=QWEN3_PATH)
    print(f"Qwen3 DL完了: {QWEN3_PATH}")
else:
    print(f"Qwen3 DL済み: {QWEN3_PATH}")

print("全モデル準備完了!")


## 4. 関数定義 (自動・定義するだけ)

データセット前処理、TOML設定ファイル作成、学習実行、およびDriveへのLoRA保存関数を定義します。

In [ ]:
import glob, os, re, shutil, subprocess, sys, zipfile

ENV = os.environ.copy()
ENV["PYTHONPATH"] = f"/content/sd-scripts:{ENV.get('PYTHONPATH', '')}"

META_TAGS = {"character name", "character", "name", "symbols", "text", "english text",
             "dated", "score", "no humans", "commentary", "translated", "artist name"}

def calc_steps(n_imgs):
    # 画像枚数から学習ステップ数を自動決定
    if n_imgs <= 10:
        return 400
    if n_imgs <= 30:
        return 1000
    if n_imgs <= 50:
        return 1000
    if n_imgs <= 100:
        return 1250
    if n_imgs <= 150:
        return 1500
    return 1800

def split_tags_and_prose(caption):
    parts = [p.strip() for p in caption.split(",")]
    prose_idx = None
    for i, p in enumerate(parts):
        words = p.split()
        if len(words) >= 4 and p[0].isupper() and not p.isupper() and not p[0].isdigit():
            prose_idx = i
            break
    if prose_idx is None:
        return parts, []
    return parts[:prose_idx], parts[prose_idx:]

def fix_caption(caption, trigger):
    tags, prose = split_tags_and_prose(caption)
    if trigger and trigger in tags:
        tags = [t for t in tags if t != trigger]
    tags = [t for t in tags if t.lower() not in META_TAGS]
    keep = []
    for t in tags:
        tlow = t.lower()
        if len(t.split()) == 1 and any(tlow in o.lower().split() and o.lower() != tlow for o in tags):
            continue
        keep.append(t)
    seen, deduped = set(), []
    for t in keep:
        if t.lower() not in seen:
            seen.add(t.lower())
            deduped.append(t)
    return ", ".join([trigger] + prose + deduped)

def prepare_dataset(zip_path, trigger):
    assert os.path.exists(zip_path), f"ZIPが見つかりません: {zip_path}"
    os.system("rm -rf /content/dataset")
    os.makedirs("/content/dataset", exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall("/content/dataset")
    for f in glob.glob("/content/dataset/**/*", recursive=True):
        if os.path.isfile(f) and os.path.dirname(f) != "/content/dataset":
            shutil.move(f, os.path.join("/content/dataset", os.path.basename(f)))
    imgs = sorted(glob.glob("/content/dataset/*.png") + glob.glob("/content/dataset/*.jpg")
                  + glob.glob("/content/dataset/*.jpeg") + glob.glob("/content/dataset/*.webp"))
    caps = sorted(glob.glob("/content/dataset/*.txt"))
    assert len(imgs) >= 10, f"画像が少なすぎます ({len(imgs)}枚, 10枚以上推奨)"
    changed = 0
    for txt in caps:
        raw = open(txt, encoding="utf-8", errors="ignore").read().strip()
        fixed = fix_caption(raw, trigger)
        if fixed != raw:
            changed += 1
        open(txt, "w", encoding="utf-8").write(fixed + "\n")
    print(f"データセット準備OK: 画像{len(imgs)}枚 / キャプション{len(caps)}個 (修正{changed}個) / trigger='{trigger}'")
    return len(imgs)

def make_dataset_toml(res=1024):
    toml_path = "/content/dataset.toml"
    toml_content = f"""[general]
shuffle_caption = false
keep_tokens = 1

[[datasets]]
resolution = [{res}, {res}]
batch_size = 1
enable_bucket = true
min_bucket_reso = 512
max_bucket_reso = {res}
bucket_reso_steps = 64

  [[datasets.subsets]]
  image_dir = "/content/dataset"
  caption_extension = ".txt"
  num_repeats = 1
"""
    with open(toml_path, "w", encoding="utf-8") as f:
        f.write(toml_content)
    print(f"データセットTOML作成OK: {toml_path} (解像度 {res}x{res}, bucketing有効)")
    return toml_path

def train_network(trigger, toml_path, steps, lr, rank):
    print(f"--- Anima LoRA学習開始: {steps} steps / lr {lr} / rank {rank} ---")
    out_dir = f"/content/output/{trigger}"
    os.makedirs(out_dir, exist_ok=True)

    cmd = [
        "accelerate", "launch",
        "--num_cpu_threads_per_process", "1",
        "--mixed_precision", "bf16",
        "/content/sd-scripts/anima_train_network.py",
        "--pretrained_model_name_or_path", DIT_PATH,
        "--qwen3", QWEN3_PATH,
        "--vae", VAE_PATH,
        "--dataset_config", toml_path,
        "--output_dir", out_dir,
        "--output_name", trigger,
        "--save_model_as", "safetensors",
        "--save_precision", "bf16",
        "--network_module", "networks.lora_anima",
        "--network_dim", str(rank),
        "--network_alpha", str(rank),
        "--network_train_unet_only",
        "--learning_rate", str(lr),
        "--optimizer_type", "AdamW8bit",
        "--lr_scheduler", "constant",
        "--timestep_sample_method", "logit_normal",
        "--discrete_flow_shift", "3.0",
        "--mixed_precision", "bf16",
        "--gradient_checkpointing",
        "--cache_latents",
        "--cache_text_encoder_outputs",
        "--max_train_steps", str(steps),
        "--save_every_n_steps", "250",
    ]
    p = subprocess.Popen(
        cmd,
        env=ENV,
        cwd="/content/sd-scripts",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in iter(p.stdout.readline, ""):
        print(line, end="")
    p.stdout.close()
    ret_code = p.wait()
    if ret_code != 0:
        raise RuntimeError(f"Anima LoRA学習が失敗しました (exit code: {ret_code})")

def deliver(trigger, steps, lr, sweep_min=None):
    out_dir = OUTPUT_DIR
    os.makedirs(out_dir, exist_ok=True)
    import datetime
    jst = datetime.timezone(datetime.timedelta(hours=9))
    dstr = datetime.datetime.now(jst).strftime("%Y%m%d")

    files = sorted(glob.glob(f"/content/output/{trigger}/*.safetensors"))
    assert files, f"学習出力ファイルが見つかりません: /content/output/{trigger}"

    delivered = []
    for f in files:
        base = os.path.basename(f)
        m = re.search(r"-step0*(\d+)\.safetensors$", base)
        if m:
            step = int(m.group(1))
            if sweep_min is not None and step < sweep_min:
                continue
            dst = os.path.join(out_dir, f"{trigger}_{dstr}_{step}steps_lr{lr}.safetensors")
        elif base == f"{trigger}.safetensors":
            dst = os.path.join(out_dir, f"{trigger}_{dstr}_{steps}steps_lr{lr}.safetensors")
        else:
            dst = os.path.join(out_dir, f"{trigger}_{dstr}_{base}")
        shutil.copy(f, dst)
        delivered.append(os.path.basename(dst))
        print("Drive保存:", os.path.basename(dst), "(%.1f MB)" % (os.path.getsize(f) / 1e6))
    print("納品数:", len(delivered))
    return delivered

print("関数定義OK (calc_steps / prepare_dataset / make_dataset_toml / train_network / deliver)")


## 5. 全ジョブ実行 (自動・本丸)

JOBSの全データセットを順番に: **展開 → キャプション修正 → TOML生成 → 学習 → Drive納品 → ZIPをdoneへ移動** します。
- 1ジョブがエラーしても残りは続行し、最後に失敗ジョブを表示します
- 学習完了したZIPは自動的に `MyDrive/anima/done/` に退避されます
- 再実行時は、完了済みジョブは自動でスキップされます


In [ ]:
import os, shutil, traceback, torch

FORCE_RERUN = False

props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1e9
gpu = torch.cuda.get_device_name(0)
print(f"GPU: {gpu} / VRAM: {vram_gb:.1f}GB")

res = RESOLUTION
DONE = "/content/jobs_done"
os.makedirs(DONE, exist_ok=True)

N = len(JOBS)
failed = []

for i, job in enumerate(JOBS, 1):
    trig, zip_ = job["trigger"], job["zip"]
    marker = os.path.join(DONE, f"{trig}.done")
    if os.path.exists(marker) and not FORCE_RERUN:
        print(f"\nSKIP: {trig} は完了済みです (再学習したい場合は FORCE_RERUN = True に設定してください)")
        continue

    print(f"\n{'='*64}\nJOB {i}/{N}: {trig} ({zip_.rsplit('/', 1)[-1]})\n{'='*64}")
    os.system(f"rm -rf /content/output/{trig}")

    try:
        n_imgs = prepare_dataset(zip_, trig)
        steps = job.get("steps") or STEPS or calc_steps(n_imgs)
        lr = job.get("lr") or LR
        rank = job.get("rank") or RANK
        print(f"→ ステップ数: {steps} (画像 {n_imgs} 枚) / lr: {lr} / rank: {rank}")

        toml_path = make_dataset_toml(res)
        train_network(trig, toml_path, steps, lr, rank)
        delivered = deliver(trig, steps, lr, sweep_min=job.get("sweep_min"))

        # 正常終了したZIPを done フォルダへ移動
        if os.path.exists(zip_):
            done_dest = os.path.join(DONE_DIR, os.path.basename(zip_))
            shutil.move(zip_, done_dest)
            print(f"📦 完了ZIPを移動: {os.path.basename(zip_)} -> {DONE_DIR}")

        with open(marker, "w") as f:
            f.write("done")
        print(f"JOB_DONE: {trig} ({i}/{N})")
    except Exception as e:
        print(f"❌ エラー発生: {trig}: {e}")
        traceback.print_exc()
        failed.append(trig)

print("\n" + "="*64)
if failed:
    print(f"⚠️ 失敗したジョブがあります: {failed}")
    print("→ このセルを再実行すると、失敗ジョブだけクリーンにやり直します")
else:
    print("🎉 ALL_JOBS_DONE: 全ジョブが正常に完了しました！")
print("="*64)


## 6. ComfyUI / LanPaint での利用方法

完成したLoRAは `MyDrive/anima/outputs/` に自動保存されています。

1. **ComfyUI への配置**:
   - Google Drive からダウンロードし、`ComfyUI/models/loras/` に配置します。
2. **ワークフローでの読み込み**:
   - 標準の `LoraLoaderModelOnly` または `LoraLoader` ノードでモデルに接続します。
   - （`sd-scripts` の出力はキーに `net.` プレフィックスが付いており、ComfyUI でそのまま認識されます）
3. **LanPaint との連携**:
   - Anima ベースモデルと今回学習した LoRA を接続した状態で LanPaint インペイントノードを通すことで、高精度なキャラクター維持・画風修正が可能です。
4. **推奨強度 (Strength)**:
   - キャラクター・画風: `0.7 〜 1.0` 推奨
